# HydroGraphFM: Reach Embedding Space Analysis

This notebook extracts the reach latent embeddings from the pretrained GNN-Transformer encoder, projects them into 2D space using PCA, t-SNE, and UMAP, and visualizes them colored by Strahler Stream Order.

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import os

from train_pretrain import HST_PAGT_Encoder, compute_positional_encodings

In [ ]:
# Setup routing graph topology
df_meta = pd.read_csv('metadata_reaches.csv')
df_pre = pd.read_parquet('hydrographfm_pretrain_full.parquet')

num_reaches = 483
edges_fwd = []
edges_bwd = []
for idx, row in df_meta.iterrows():
    u = int(row['Reach_ID'])
    d = row['Downstream_ID']
    if not pd.isna(d):
        v = int(d)
        if 1 <= v <= num_reaches:
            edges_fwd.append((u - 1, v - 1))
            edges_bwd.append((v - 1, u - 1))
edge_index = torch.tensor(edges_fwd + edges_bwd, dtype=torch.long).t().contiguous()
pos_enc = compute_positional_encodings(df_meta, df_pre)

meta_cols = ['Sample_ID', 'Time_Lag', 'Date', 'Reach_ID', 'Subbasin_ID', 'Upstream_ID', 'Downstream_ID']
feat_cols = [c for c in df_pre.columns if c not in meta_cols]

X_raw = df_pre[feat_cols].values
num_samples = 205
X_train = X_raw.reshape(num_reaches, num_samples, 24, -1)
X_train = np.transpose(X_train, (1, 0, 2, 3))

print("Features shape:", X_train.shape)

In [ ]:
# Load encoder weights
encoder = HST_PAGT_Encoder(len(feat_cols), 64, edge_index, pos_enc, conv_type='gat')
if os.path.exists('hydrographfm_encoder_v1.pt'):
    encoder.load_state_dict(torch.load('hydrographfm_encoder_v1.pt'))
    print("Pretrained GAT encoder loaded successfully!")
encoder.eval()

# Extract embeddings
emb_list = []
with torch.no_grad():
    for b in range(int(np.ceil(num_samples / 16))):
        batch_data = X_train[b*16 : (b+1)*16]
        if len(batch_data) == 0:
            continue
        x = torch.tensor(batch_data, dtype=torch.float32)
        h = encoder(x)
        emb_list.append(h[:, :, 23, :].numpy())

H = np.concatenate(emb_list, axis=0)
H_nodes = H.mean(axis=0) # Mean pool over temporal samples
print("Node embeddings shape:", H_nodes.shape)

In [ ]:
# Compute dimensional reduction projections
pca = PCA(n_components=2, random_state=42)
z_pca = pca.fit_transform(H_nodes)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
z_tsne = tsne.fit_transform(H_nodes)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
z_umap = reducer.fit_transform(H_nodes)

In [ ]:
# Reconstruct unscaled Strahler Stream Order for plotting
up_map = {}
for idx, row in df_meta.iterrows():
    u = int(row['Reach_ID'])
    up = row['Upstream_ID']
    if pd.isna(up):
        up_map[u] = []
    else:
        up_map[u] = [int(x) for x in str(up).split(',') if x.strip()]

orders_raw = np.zeros(num_reaches, dtype=np.int32)
orders_dict = {}
def get_order_raw(node):
    if node in orders_dict:
        return orders_dict[node]
    ups = up_map.get(node, [])
    if not ups:
        orders_dict[node] = 1
        return 1
    up_orders = [get_order_raw(u) for u in ups]
    max_order = max(up_orders)
    if up_orders.count(max_order) > 1:
        orders_dict[node] = max_order + 1
    else:
        orders_dict[node] = max_order
    return orders_dict[node]

for r in range(1, num_reaches + 1):
    orders_raw[r-1] = get_order_raw(r)

In [ ]:
# Plot projections
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(x=z_pca[:, 0], y=z_pca[:, 1], hue=orders_raw, palette='viridis', legend='full', ax=axes[0])
axes[0].set_title('PCA Projection')
axes[0].grid(True)

sns.scatterplot(x=z_tsne[:, 0], y=z_tsne[:, 1], hue=orders_raw, palette='viridis', legend='full', ax=axes[1])
axes[1].set_title('t-SNE Projection')
axes[1].grid(True)

sns.scatterplot(x=z_umap[:, 0], y=z_umap[:, 1], hue=orders_raw, palette='viridis', legend='full', ax=axes[2])
axes[2].set_title('UMAP Projection')
axes[2].grid(True)

plt.suptitle('HydroGraphFM: Reach Embedding Space Projections', fontsize=14)
plt.tight_layout()
plt.show()